This is a raw dataset i prepared myself for machine learning using python,pandas and scikit-learnthis dataset is a synthetic customer churn dataset,with missing values ,duplicated rows and outliers added on top so i could practice handling each one.

I first of all load the cvs file and look and it sharp,column type , summary statistics, missing values and duplicates.this way i understand what needs fixing before the data is used for a model

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("raw_customers.csv")
print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe()

Shape: (1010, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    1010 non-null   int64  
 1   age            930 non-null    float64
 2   income         891 non-null    float64
 3   tenure_months  1010 non-null   float64
 4   city           960 non-null    object 
 5   gender         1010 non-null   object 
 6   plan           980 non-null    object 
 7   churned        1010 non-null   int64  
dtypes: float64(3), int64(2), object(3)
memory usage: 63.3+ KB
customer_id        0
age               80
income           119
tenure_months      0
city              50
gender             0
plan              30
churned            0
dtype: int64
Duplicate rows: 10


,customer_id,age,income,tenure_months,churned
count,1010.000000,930.000000,8.910000e+02,1010.000000,1010.000000
mean,501.104950,43.640860,2.763957e+05,59.601980,0.325743
std,288.945177,14.984816,3.665523e+05,34.038892,0.468884
min,1.000000,18.000000,-7.835700e+04,1.000000,0.000000
25%,252.250000,30.000000,1.913435e+05,29.000000,0.000000
50%,501.500000,44.000000,2.525220e+05,59.000000,0.000000
75%,750.750000,56.000000,3.093985e+05,89.000000,1.000000
max,1000.000000,69.000000,5.000000e+06,119.000000,1.000000


cleaning and spliting the data

## 2. Cleaning and splitting the data

so i was able to removed the 10 duplicate rows and dropped customer_id, because an ID number has no predictive value. Then I also  split the data into 80% training and 20% testing sets before doing anything else.                                           Splitting first matters: if I calculated medians or scaling values on the whole dataset, information from the test set would leak into training. so
I used stratify so the churn rate is about the same in both sets.

In [2]:
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop(columns="customer_id")

X = df.drop(columns="churned")
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test = X_train.copy(), X_test.copy()
print(X_train.shape, X_test.shape)

(800, 6) (200, 6)


## 3. Handling missing values

as we can see four columns had missing values: age, income, city and plan. For the numeric columns (age, income, tenure_months) I filled gaps with the median, because income has extreme outliers that would pull the mean upward. so For the categorical columns (city, gender, plan) I used the most frequent value, since text can't be averaged. I calculated every fill value from the training set only and applied the same values to the test set, so the test data stays unseen.

In [3]:
num_cols = ["age", "income", "tenure_months"]
cat_cols = ["city", "gender", "plan"]

for c in num_cols:
    med = X_train[c].median()
    X_train[c] = X_train[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

for c in cat_cols:
    mode = X_train[c].mode()[0]
    X_train[c] = X_train[c].fillna(mode)
    X_test[c] = X_test[c].fillna(mode)

print("Missing values left:", X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

Missing values left: 0 0


## 4. Encoding categorical variables

Machine learning models need numbers, so the three text columns had to be converted. The plan column has a natural order (Basic, Standard, Premium), so I used ordinal encoding and mapped them to 0, 1 and 2. so the city and gender columns have no order, so I used one-hot encoding, which creates a separate 0/1 column for each value. This stops the model from treating one city as "bigger" than another. The encoder learned its categories from the training set only.

In [4]:
# Ordinal encoding for plan
plan_map = {"Basic": 0, "Standard": 1, "Premium": 2}
X_train["plan"] = X_train["plan"].map(plan_map)
X_test["plan"] = X_test["plan"].map(plan_map)

# One-hot encoding for city and gender
ohe_cols = ["city", "gender"]
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe.fit(X_train[ohe_cols])
new_cols = ohe.get_feature_names_out(ohe_cols)

train_ohe = pd.DataFrame(ohe.transform(X_train[ohe_cols]), columns=new_cols, index=X_train.index)
test_ohe = pd.DataFrame(ohe.transform(X_test[ohe_cols]), columns=new_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=ohe_cols), train_ohe], axis=1)
X_test = pd.concat([X_test.drop(columns=ohe_cols), test_ohe], axis=1)

print(X_train.shape, X_test.shape)
print(list(X_train.columns))
X_train.head()

(800, 11) (200, 11)
['age', 'income', 'tenure_months', 'plan', 'city_Abuja', 'city_Kano', 'city_Lagos', 'city_Minna', 'city_Port Harcourt', 'gender_Female', 'gender_Male']


,age,income,tenure_months,plan,city_Abuja,city_Kano,city_Lagos,city_Minna,city_Port Harcourt,gender_Female,gender_Male
653,59.0,235156.0,77.0,1,0.0,0.0,0.0,1.0,0.0,0.0,1.0
111,58.0,413917.0,95.0,0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
694,22.0,105981.0,46.0,2,1.0,0.0,0.0,0.0,0.0,0.0,1.0
217,23.0,167636.0,49.0,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
743,32.0,168919.0,66.0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## 5. Scaling numeric features

Income is in the hundreds of thousands, while age and tenure are small numbers. so i used StandardScaler to put age, income and tenure on the same scale (mean 0, standard deviation 1), fitted on the training set only.

In [5]:
scale_cols = ["age", "income", "tenure_months"]
scaler = StandardScaler().fit(X_train[scale_cols])

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_cols] = pd.DataFrame(
    scaler.transform(X_train[scale_cols]), columns=scale_cols, index=X_train.index
)
X_test_scaled[scale_cols] = pd.DataFrame(
    scaler.transform(X_test[scale_cols]), columns=scale_cols, index=X_test.index
)

print(X_train_scaled[scale_cols].describe().loc[["mean", "std"]].round(3))
print()
print(X_test_scaled[scale_cols].describe().loc[["mean", "std"]].round(3))
X_train_scaled.head()

        age  income  tenure_months
mean -0.000  -0.000         -0.000
std   1.001   1.001          1.001

        age  income  tenure_months
mean  0.017  -0.102          0.054
std   0.922   0.214          1.034


,age,income,tenure_months,plan,city_Abuja,city_Kano,city_Lagos,city_Minna,city_Port Harcourt,gender_Female,gender_Male
653,1.052239,-0.122258,0.521252,1,0.0,0.0,0.0,1.0,0.0,0.0,1.0
111,0.983567,0.343351,1.053972,0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
694,-1.488655,-0.458714,-0.396210,2,1.0,0.0,0.0,0.0,0.0,0.0,1.0
217,-1.419983,-0.298124,-0.307424,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
743,-0.801927,-0.294782,0.195701,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## 6. Final check and saved files

so i checked that there are no missing values, all columns are numeric, and the churn rate is similar in both sets. The processed data is saved as train_processed.csv and test_processed.csv.

In [6]:
print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)
print("Missing values:", X_train_scaled.isnull().sum().sum(), X_test_scaled.isnull().sum().sum())
print("Non-numeric columns:", list(X_train_scaled.select_dtypes(exclude="number").columns))
print("Churn rate train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))

X_train_scaled.assign(churned=y_train).to_csv("train_processed.csv", index=False)
X_test_scaled.assign(churned=y_test).to_csv("test_processed.csv", index=False)
print("Saved train_processed.csv and test_processed.csv")

Train: (800, 11) | Test: (200, 11)
Missing values: 0 0
Non-numeric columns: []
Churn rate train/test: 0.326 0.325
Saved train_processed.csv and test_processed.csv


everything checks out and the data is ready for a model